In [8]:
# --- BIBLIOTHÈQUES STANDARDS (Python) ---
import re               # Pour les expressions régulières (supprimer les URLs)
import string           # Pour accéder à la liste complète de ponctuation

# --- NLP : PRÉTRAITEMENT & RACINISATION (NLTK) ---
import nltk             # La base pour la tokenization et les stopwords
from nltk.tokenize import word_tokenize        # Pour découper le texte en mots
from nltk.corpus import stopwords              # Pour filtrer les mots vides (the, is, etc.)
from nltk.stem.porter import PorterStemmer    # Pour le stemming (racinisation)

# --- NLP : ANALYSE AVANCÉE (spaCy & TextBlob) ---
import spacy            # Pour la lemmatisation et le tagging haute précision
from textblob import TextBlob # Pour l'analyse de sentiment et les corrections rapides

# --- OUTILS SPÉCIALISÉS ---
import inflect          # Pour convertir les chiffres (3) en mots (three)
import emoji            # Pour détecter et supprimer les emojis 

# Pour NLTK
nltk.download('punkt')      # Modèle de tokenisation
nltk.download('stopwords')  # Liste des mots vides

# Pour spaCy (à faire dans le terminal)
!pip install python3 -m spacy download en_core_web_sm

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lucas.fandinoperez\AppData\Roaming\nltk_data.
[nltk_data]     ..
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lucas.fandinoperez\AppData\Roaming\nltk_data.
[nltk_data]     ..
[nltk_data]   Package stopwords is already up-to-date!

Usage:   
  pip install [options] <requirement specifier> [package-index-options] ...
  pip install [options] -r <requirements file> [package-index-options] ...
  pip install [options] [-e] <vcs project url> ...
  pip install [options] [-e] <local project path> ...
  pip install [options] <archive url/path> ...

no such option: -m


In [9]:
import pandas as pd

df = pd.read_csv('SMS_test.csv', encoding='latin-1')



In [10]:


df.isnull().sum()

S. No.          0
Message_body    0
Label           0
dtype: int64

In [11]:
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   S. No.        125 non-null    int64
 1   Message_body  125 non-null    str  
 2   Label         125 non-null    str  
dtypes: int64(1), str(2)
memory usage: 3.1 KB


,S. No.
count,125.000000
mean,63.000000
std,36.228442
min,1.000000
25%,32.000000
50%,63.000000
75%,94.000000
max,125.000000


In [12]:
df.head(10)

,S. No.,Message_body,Label
0,1,"UpgrdCentre Orange customer, you may now claim...",Spam
1,2,"Loan for any purpose £500 - £75,000. Homeowner...",Spam
2,3,Congrats! Nokia 3650 video camera phone is you...,Spam
3,4,URGENT! Your Mobile number has been awarded wi...,Spam
4,5,Someone has contacted our dating service and e...,Spam
5,6,Send a logo 2 ur lover - 2 names joined by a h...,Spam
6,7,FREE entry into our £250 weekly competition ju...,Spam
7,8,100 dating service cal;l 09064012103 box334sk38ch,Spam
8,9,FREE RINGTONE text FIRST to 87131 for a poly o...,Spam
9,10,4mths half price Orange line rental & latest c...,Spam


In [13]:
messages = df["Message_body"]

messages.head()

0    UpgrdCentre Orange customer, you may now claim...
1    Loan for any purpose £500 - £75,000. Homeowner...
2    Congrats! Nokia 3650 video camera phone is you...
3    URGENT! Your Mobile number has been awarded wi...
4    Someone has contacted our dating service and e...
Name: Message_body, dtype: str

# Text Normalization (Lowercasing, Handling contractions, Spell schecking)


In [14]:
# Lowercasing

# on définit notre fonction lowercase
def text_lower(text):
    return text.str.lower()


# on utilise notre fct
text = text_lower(messages)




In [15]:
print(text)

# on check si TOUTE la colonne est déjà en minuscules
est_propre = text.equals(text.str.lower())
print(f"\nLa colonne est-elle totalement en minuscules ? {est_propre}")



0      upgrdcentre orange customer, you may now claim...
1      loan for any purpose £500 - £75,000. homeowner...
2      congrats! nokia 3650 video camera phone is you...
3      urgent! your mobile number has been awarded wi...
4      someone has contacted our dating service and e...
                             ...                        
120    7 wonders in my world 7th you 6th ur style 5th...
121    try to do something dear. you read something f...
122    sun ah... thk mayb can if dun have anythin on....
123    symptoms when u are in love: "1.u like listeni...
124    great. have a safe trip. dont panic surrender ...
Name: Message_body, Length: 125, dtype: str

La colonne est-elle totalement en minuscules ? True


In [16]:
# dealing with contractions

import contractions

def fix_contractions(text):
    if not isinstance(text, str):
        return text
    
    expanded_text = contractions.fix(text)
    return expanded_text

text = fix_contractions(text)


In [17]:
print(text)

0      upgrdcentre orange customer, you may now claim...
1      loan for any purpose £500 - £75,000. homeowner...
2      congrats! nokia 3650 video camera phone is you...
3      urgent! your mobile number has been awarded wi...
4      someone has contacted our dating service and e...
                             ...                        
120    7 wonders in my world 7th you 6th ur style 5th...
121    try to do something dear. you read something f...
122    sun ah... thk mayb can if dun have anythin on....
123    symptoms when u are in love: "1.u like listeni...
124    great. have a safe trip. dont panic surrender ...
Name: Message_body, Length: 125, dtype: str


In [18]:
def fix_contractions(text_list):
    return [contractions.fix(message) for message in text_list]


text = fix_contractions(text)
print(text[0]) # Devrait afficher "...you may now claim..."

upgrdcentre orange customer, you may now claim your free camera phone upgrade for your loyalty. call now on 0207 153 9153. offer ends 26th july. t&c's apply. opt-out available


In [19]:
# spell checking 
!pip install autocorrect
from autocorrect import Speller


def correct(text_list, lang="en"):
    "Correct spellin using autocorrect for english."
    spell = Speller(lang=lang)
    return [spell(message) for message in text_list]

text = correct(text)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
print(text)

["upgrdcentre orange customer, you may now claim your free camera phone upgrade for your loyalty. call now on 0207 153 9153. offer ends 26th july. t&c's apply. opt-out available", "loan for any purpose £500 - £75,000. homeowners + tenants welcome. have you been previously refused? we can still help. call free 0800 1956669 or text back 'help'", 'contrast! nokia 3650 video camera phone is your call 09066382422 calls cost 150ppm ave call 3mins vary from mobiles 16+ close 300603 post bcm4284 lon wc1n3xx', 'urgent! your mobile number has been awarded with a £2000 prize guaranteed. call 09058094455 from land line. claim 3030. valid 12hrs only', 'someone has contacted our dating service and entered your phone because they fancy you! to find out who it is call from a landing 09111032124 . box12n146tf150p', 'send a logo 2 you are lover - 2 names joined by a heart. txt love name1 name2 mono eg love adam eve 07123456789 to 87077 yahoo! box36504w45wq xeno 4 no ads 150p', 'free entry into our £250 

# NOISE REMOVAL

In [21]:
### Noise Removal (Removing numbers/digits, Punctuation & Special Characters, Handling double whitespace from text, and Removal of URLs)

# removal of URLs
import re


combined_url_pattern = re.compile(r'https?://\S+|www\.\S+')

def remove_urls(text_list):
    
    return [combined_url_pattern.sub('<weblink>', message) for message in text_list]

# Application
text = remove_urls(text)

In [22]:
print(text)

["upgrdcentre orange customer, you may now claim your free camera phone upgrade for your loyalty. call now on 0207 153 9153. offer ends 26th july. t&c's apply. opt-out available", "loan for any purpose £500 - £75,000. homeowners + tenants welcome. have you been previously refused? we can still help. call free 0800 1956669 or text back 'help'", 'contrast! nokia 3650 video camera phone is your call 09066382422 calls cost 150ppm ave call 3mins vary from mobiles 16+ close 300603 post bcm4284 lon wc1n3xx', 'urgent! your mobile number has been awarded with a £2000 prize guaranteed. call 09058094455 from land line. claim 3030. valid 12hrs only', 'someone has contacted our dating service and entered your phone because they fancy you! to find out who it is call from a landing 09111032124 . box12n146tf150p', 'send a logo 2 you are lover - 2 names joined by a heart. txt love name1 name2 mono eg love adam eve 07123456789 to 87077 yahoo! box36504w45wq xeno 4 no ads 150p', 'free entry into our £250 

In [23]:
# removing numbers or digits

def remove_numbers(text_list):
    
    return [re.sub(r'\d+', '<masked number>', message) for message in text_list]





In [24]:
text = remove_numbers(text)

In [25]:
print(text)

["upgrdcentre orange customer, you may now claim your free camera phone upgrade for your loyalty. call now on <masked number> <masked number> <masked number>. offer ends <masked number>th july. t&c's apply. opt-out available", "loan for any purpose £<masked number> - £<masked number>,<masked number>. homeowners + tenants welcome. have you been previously refused? we can still help. call free <masked number> <masked number> or text back 'help'", 'contrast! nokia <masked number> video camera phone is your call <masked number> calls cost <masked number>ppm ave call <masked number>mins vary from mobiles <masked number>+ close <masked number> post bcm<masked number> lon wc<masked number>n<masked number>xx', 'urgent! your mobile number has been awarded with a £<masked number> prize guaranteed. call <masked number> from land line. claim <masked number>. valid <masked number>hrs only', 'someone has contacted our dating service and entered your phone because they fancy you! to find out who it i

# tokenization

In [26]:
import nltk

# modèle de segmentation en phrases et mots
nltk.download('punkt')

# Si tu utilises une version très récente, ce paquet peut aussi être requis
nltk.download('punkt_tab')


def tokenize_text(text):
    # Découpe le texte en composants plus petits (tokens)
    return nltk.word_tokenize(text)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lucas.fandinoperez\AppData\Roaming\nltk_data.
[nltk_data]     ..
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lucas.fandinoperez\AppData\Roaming\nltk_data.
[nltk_data]     ..
[nltk_data]   Package punkt_tab is already up-to-date!


In [29]:
tokenized = [tokenize_text(msg) for msg in text]
print("Result from tokenization:", tokenized)

Result from tokenization: [['upgrdcentre', 'orange', 'customer', ',', 'you', 'may', 'now', 'claim', 'your', 'free', 'camera', 'phone', 'upgrade', 'for', 'your', 'loyalty', '.', 'call', 'now', 'on', '<', 'masked', 'number', '>', '<', 'masked', 'number', '>', '<', 'masked', 'number', '>', '.', 'offer', 'ends', '<', 'masked', 'number', '>', 'th', 'july', '.', 't', '&', 'c', "'s", 'apply', '.', 'opt-out', 'available'], ['loan', 'for', 'any', 'purpose', '£', '<', 'masked', 'number', '>', '-', '£', '<', 'masked', 'number', '>', ',', '<', 'masked', 'number', '>', '.', 'homeowners', '+', 'tenants', 'welcome', '.', 'have', 'you', 'been', 'previously', 'refused', '?', 'we', 'can', 'still', 'help', '.', 'call', 'free', '<', 'masked', 'number', '>', '<', 'masked', 'number', '>', 'or', 'text', 'back', "'help", "'"], ['contrast', '!', 'nokia', '<', 'masked', 'number', '>', 'video', 'camera', 'phone', 'is', 'your', 'call', '<', 'masked', 'number', '>', 'calls', 'cost', '<', 'masked', 'number', '>', '

# stopwords removal

In [36]:
def remove_stopwords(text, lang='english'):
    # 1. Charger la liste des mots vides pour la langue choisie
    stop_words = set(stopwords.words(lang))
    
    # 2. Filtrer : on ne garde que les mots qui ne sont pas dans la liste
    # (en mettant tout en minuscule pour la comparaison)
    filtered_text = [w for w in text if w.lower() not in stop_words]
    
    return filtered_text




In [ ]:
text = [remove_stopwords(msg) for msg in tokenized]



In [38]:
print(text)

[['upgrdcentre', 'orange', 'customer', ',', 'may', 'claim', 'free', 'camera', 'phone', 'upgrade', 'loyalty', '.', 'call', '<', 'masked', 'number', '>', '<', 'masked', 'number', '>', '<', 'masked', 'number', '>', '.', 'offer', 'ends', '<', 'masked', 'number', '>', 'th', 'july', '.', '&', 'c', "'s", 'apply', '.', 'opt-out', 'available'], ['loan', 'purpose', '£', '<', 'masked', 'number', '>', '-', '£', '<', 'masked', 'number', '>', ',', '<', 'masked', 'number', '>', '.', 'homeowners', '+', 'tenants', 'welcome', '.', 'previously', 'refused', '?', 'still', 'help', '.', 'call', 'free', '<', 'masked', 'number', '>', '<', 'masked', 'number', '>', 'text', 'back', "'help", "'"], ['contrast', '!', 'nokia', '<', 'masked', 'number', '>', 'video', 'camera', 'phone', 'call', '<', 'masked', 'number', '>', 'calls', 'cost', '<', 'masked', 'number', '>', 'ppm', 'ave', 'call', '<', 'masked', 'number', '>', 'mins', 'vary', 'mobiles', '<', 'masked', 'number', '>', '+', 'close', '<', 'masked', 'number', '>',

# Stemming

In [41]:
p = inflect.engine()
stemmer = PorterStemmer()

def stem_words(text):
    # Stemming : réduit chaque mot à sa racine (ex: scientific -> scientif)
    stems = [stemmer.stem(word) for word in text]

    return stems

stem_text = [stem_words(msg) for msg in text]

print(stem_text)

[['upgrdcentr', 'orang', 'custom', ',', 'may', 'claim', 'free', 'camera', 'phone', 'upgrad', 'loyalti', '.', 'call', '<', 'mask', 'number', '>', '<', 'mask', 'number', '>', '<', 'mask', 'number', '>', '.', 'offer', 'end', '<', 'mask', 'number', '>', 'th', 'juli', '.', '&', 'c', "'s", 'appli', '.', 'opt-out', 'avail'], ['loan', 'purpos', '£', '<', 'mask', 'number', '>', '-', '£', '<', 'mask', 'number', '>', ',', '<', 'mask', 'number', '>', '.', 'homeown', '+', 'tenant', 'welcom', '.', 'previous', 'refus', '?', 'still', 'help', '.', 'call', 'free', '<', 'mask', 'number', '>', '<', 'mask', 'number', '>', 'text', 'back', "'help", "'"], ['contrast', '!', 'nokia', '<', 'mask', 'number', '>', 'video', 'camera', 'phone', 'call', '<', 'mask', 'number', '>', 'call', 'cost', '<', 'mask', 'number', '>', 'ppm', 'ave', 'call', '<', 'mask', 'number', '>', 'min', 'vari', 'mobil', '<', 'mask', 'number', '>', '+', 'close', '<', 'mask', 'number', '>', 'post', 'bcm', '<', 'mask', 'number', '>', 'lon', 'wc

# Lemmatization

In [44]:
nlp = spacy.load("en_core_web_sm")

def lemmatize_text(text):
    text = " ".join(text)
    doc = nlp(text)
    # On récupère le lemme de chaque token
    return ' '.join([token.lemma_ for token in doc])


lema_text = [lemmatize_text(msg) for msg in text]


In [45]:
print(lema_text)

["upgrdcentre orange customer , may claim free camera phone upgrade loyalty . call < mask number > < mask number > < mask number > . offer end < mask number > th july . & c 's apply . opt - out available", "loan purpose £ < mask number > - £ < mask number > , < mask number > . homeowner + tenant welcome . previously refuse ? still help . call free < mask number > < mask number > text back ' help '", 'contrast ! nokia < mask number > video camera phone call < mask number > call cost < mask number > ppm ave call < mask number > min vary mobile < mask number > + close < mask number > post bcm < mask number > lon wc < mask number > n < mask number > xx', 'urgent ! mobile number award £ < mask number > prize guarantee . call < mask number > land line . claim < mask number > . valid < mask number > hrs', 'someone contact date service enter phone fancy ! find call landing < mask number > . box < mask number > n < mask number > tf < mask number > p', 'send logo < mask number > lover - < mask n

# exercise 3 

In [46]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))
nlp = spacy.load("en_core_web_sm")

# --- FONCTIONS DE PRÉPARATION ---

def remove_noise(text):
    # Supprime ponctuation et caractères spéciaux
    text = re.sub(r'[^\w\s]', '', text)
    # Supprime les balises spécifiques si présentes (ex: <mask number>)
    text = text.replace('mask number', '').replace('weblink', '')
    return text

def lemmatize_list(word_list):
    # On transforme la liste en string pour spaCy
    doc = nlp(" ".join(word_list))
    return [token.lemma_ for token in doc]

# --- CRÉATION DES 9 COLONNES ---

# 1. & 2. Colonnes existantes (S.No, Message_body, Label)

# 3. Case Normalization
df['Case Normalization'] = df['Message_body'].str.lower()

# 4. Noise Removal
df['Noise Removal'] = df['Case Normalization'].apply(remove_noise)

# 5. Tokenization
df['Tokenization'] = df['Noise Removal'].apply(word_tokenize)

# 6. Stopwords Removal
df['Stopwords'] = df['Tokenization'].apply(
    lambda x: [word for word in x if word not in stop_words and word.isalnum()]
)

# 7. Stemming (sur la colonne Stopwords)
df['Stemming'] = df['Stopwords'].apply(
    lambda x: [stemmer.stem(word) for word in x]
)

# 8. Lemmatization (sur la colonne Stopwords - important de ne pas le faire sur le stemming)
df['Lemmatization'] = df['Stopwords'].apply(lemmatize_list)

# Affichage du résultat final (les 9 colonnes)
print(f"Nombre de colonnes : {len(df.columns)}")
df.head()

Nombre de colonnes : 9


,S. No.,Message_body,Label,Case Normalization,Noise Removal,Tokenization,Stopwords,Stemming,Lemmatization
0,1,"UpgrdCentre Orange customer, you may now claim...",Spam,"upgrdcentre orange customer, you may now claim...",upgrdcentre orange customer you may now claim ...,"[upgrdcentre, orange, customer, you, may, now,...","[upgrdcentre, orange, customer, may, claim, fr...","[upgrdcentr, orang, custom, may, claim, free, ...","[upgrdcentre, orange, customer, may, claim, fr..."
1,2,"Loan for any purpose £500 - £75,000. Homeowner...",Spam,"loan for any purpose £500 - £75,000. homeowner...",loan for any purpose 500 75000 homeowners te...,"[loan, for, any, purpose, 500, 75000, homeowne...","[loan, purpose, 500, 75000, homeowners, tenant...","[loan, purpos, 500, 75000, homeown, tenant, we...","[loan, purpose, 500, 75000, homeowner, tenant,..."
2,3,Congrats! Nokia 3650 video camera phone is you...,Spam,congrats! nokia 3650 video camera phone is you...,congrats nokia 3650 video camera phone is your...,"[congrats, nokia, 3650, video, camera, phone, ...","[congrats, nokia, 3650, video, camera, phone, ...","[congrat, nokia, 3650, video, camera, phone, c...","[congrats, nokia, 3650, video, camera, phone, ..."
3,4,URGENT! Your Mobile number has been awarded wi...,Spam,urgent! your mobile number has been awarded wi...,urgent your mobile number has been awarded wit...,"[urgent, your, mobile, number, has, been, awar...","[urgent, mobile, number, awarded, 2000, prize,...","[urgent, mobil, number, award, 2000, prize, gu...","[urgent, mobile, number, award, 2000, prize, g..."
4,5,Someone has contacted our dating service and e...,Spam,someone has contacted our dating service and e...,someone has contacted our dating service and e...,"[someone, has, contacted, our, dating, service...","[someone, contacted, dating, service, entered,...","[someon, contact, date, servic, enter, phone, ...","[someone, contact, date, service, enter, phone..."
